In [ ]:
# ============================================================
# EXERCISE 01
# ImageNet Preprocessing and Visualization
# using a Pre-trained CNN (VGG16)
# ============================================================

# -----------------------------
# 1. Import required libraries
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

# Import VGG16 pretrained model
from tensorflow.keras.applications import VGG16

# Import VGG16-specific preprocessing and prediction decoder
from tensorflow.keras.applications.vgg16 import (
    preprocess_input,
    decode_predictions
)

# Functions for loading and converting images
from tensorflow.keras.utils import img_to_array


# ============================================================
# 2. Load the pre-trained VGG16 model
# ============================================================

# weights="imagenet" means VGG16 is already trained
# on the ImageNet dataset.
model = VGG16(weights="imagenet")

print("VGG16 model loaded successfully!")


# ============================================================
# 3. Load ImageNet-derived dataset
# ============================================================

# Imagenette is a small dataset derived from ImageNet.
# It contains 10 ImageNet classes.
#
# Images are loaded directly as TensorFlow tensors.
# We do NOT create image files such as cat.jpg or dog.jpg.

dataset, info = tfds.load(
    "imagenette/320px-v2",
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=True
)

print("Dataset loaded successfully!")
print(
    "Number of images:",
    info.splits["train"].num_examples
)


# ============================================================
# 4. Randomly select 5 images from the dataset
# ============================================================

sample_dataset = dataset.shuffle(
    buffer_size=1000,
    seed=42
).take(5)


# ============================================================
# 5. Process each image
# ============================================================

# Get class names from the dataset
class_names = info.features["label"].names


for image, label in sample_dataset:

    print("\n========================================")
    print("Original Label:", class_names[label.numpy()])
    print("========================================")


    # --------------------------------------------------------
    # 5.1 Load and resize the image
    # --------------------------------------------------------

    # VGG16 expects an image of size 224 x 224 x 3.

    image = tf.image.resize(
        image,
        (224, 224)
    )

    print("Image size:", image.shape)


    # --------------------------------------------------------
    # 5.2 Convert image into a NumPy array
    # --------------------------------------------------------

    # Converts the image into numerical pixel values.

    image_array = img_to_array(image).copy()

    print("Array shape:", image_array.shape)


    # --------------------------------------------------------
    # 5.3 Add batch dimension
    # --------------------------------------------------------

    # Before:
    # (224, 224, 3)
    #
    # After:
    # (1, 224, 224, 3)

    image_batch = np.expand_dims(
        image_array,
        axis=0
    )

    print("Batch shape:", image_batch.shape)


    # --------------------------------------------------------
    # 5.4 Preprocess the image for VGG16
    # --------------------------------------------------------

    # Converts the image pixels into the format
    # expected by the pretrained VGG16 model.

    processed_image = preprocess_input(
        image_batch
    )

    print(
        "Preprocessed shape:",
        processed_image.shape
    )


    # ========================================================
    # 6. Predict the image class
    # ========================================================

    predictions = model.predict(
        processed_image,
        verbose=0
    )


    # ========================================================
    # 7. Obtain Top-5 predictions
    # ========================================================

    top_5 = decode_predictions(
        predictions,
        top=5
    )[0]


    # ========================================================
    # 8. Display Top-5 predictions and probabilities
    # ========================================================

    print("\nTop-5 Predictions:")

    for rank, (class_id, class_name, probability) in enumerate(
            top_5,
            start=1):

        print(
            rank,
            ".",
            class_name,
            "->",
            round(probability * 100, 2),
            "%"
        )


    # ========================================================
    # 9. Compare original and predicted labels
    # ========================================================

    predicted_label = top_5[0][1]

    print("\nOriginal Label :", class_names[label.numpy()])
    print("Predicted Label:", predicted_label)


    # ========================================================
    # 10. Display the input image
    # ========================================================

    plt.imshow(
        image.numpy().astype("uint8")
    )

    plt.title(
        "Original: " + class_names[label.numpy()] +
        "\nPredicted: " + predicted_label
    )

    plt.axis("off")

    plt.show()


# ============================================================
# 11. Extract Intermediate Representation
# ============================================================

# A CNN learns different features at different layers.
#
# Early layers:
#       Edges and simple patterns
#
# Middle layers:
#       Shapes and textures
#
# Deeper layers:
#       Object parts and high-level features
#
# We extract the output of VGG16's last convolutional layer.

feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer(
        "block5_conv3"
    ).output
)


# ============================================================
# 12. Load one image for feature extraction
# ============================================================

# Take one image directly from the dataset.
# No image file is created.

feature_image, feature_label = next(
    iter(
        dataset.take(1)
    )
)

image = tf.image.resize(
    feature_image,
    (224, 224)
)


# Convert image into NumPy array
# .copy() ensures the array is writable.
image_array = img_to_array(image).copy()

# Add batch dimension
image_batch = np.expand_dims(
    image_array,
    axis=0
)

# Preprocess image
processed_image = preprocess_input(
    image_batch
)


# ============================================================
# 13. Extract the intermediate feature representation
# ============================================================

features = feature_model.predict(
    processed_image,
    verbose=0
)

print("\n========================================")
print("INTERMEDIATE REPRESENTATION")
print("========================================")

print(
    "Feature map shape:",
    features.shape
)


# ============================================================
# 14. Visualize one feature map
# ============================================================

# features[0, :, :, 0] means:
#
# 0 -> first image
# : -> all rows
# : -> all columns
# 0 -> first feature channel

plt.imshow(
    features[0, :, :, 0],
    cmap="viridis"
)

plt.title(
    "Intermediate Feature Map"
)

plt.axis("off")

plt.show()


# ============================================================
# END OF PROGRAM
# ============================================================